# ⚡ Otimização de dados com PySpark — Projeto YouTube

Nesta etapa vamos comparar diferentes formas de realizar um **join** entre os dados de vídeos e comentários.

A proposta é ir além de fazer o código funcionar: vamos observar o impacto das partições, utilizar `repartition`, `coalesce`, `broadcast` e `explain`, além de filtrar e selecionar somente o que é necessário.

> **Arquivos necessários na mesma pasta deste notebook**
> - `videos-preparados.snappy.parquet`
> - `video-comments-tratados.snappy.parquet`
>
> Caso o Windows tenha acrescentado `(1)` ao nome dos arquivos, o notebook também consegue localizá-los.


## 0. Preparação do ambiente

Caso o PySpark ainda não esteja instalado no ambiente do notebook, execute a célula opcional abaixo uma única vez.


In [ ]:
# Descomente e execute somente se o PySpark ainda não estiver instalado.
# %pip install pyspark


In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.functions import broadcast

# A SparkSession é o ponto de entrada para trabalhar com PySpark.
spark = (
    SparkSession.builder
    .appName("OtimizacaoYouTube")
    .master("local[*]")
    .getOrCreate()
)

# Reduzimos a quantidade de mensagens de log para deixar a saída do notebook mais limpa.
spark.sparkContext.setLogLevel("WARN")

print("Versão do Spark:", spark.version)


## 1. Leitura do `videos-preparados.snappy.parquet`

Vamos carregar a base de vídeos no DataFrame `df_video`.


In [ ]:
BASE_DIR = Path.cwd()

# Primeiro procuramos o nome exato solicitado na atividade.
# Depois aceitamos a versão com "(1)", que pode aparecer quando um arquivo é baixado mais de uma vez.
possiveis_videos = [
    BASE_DIR / "videos-preparados.snappy.parquet",
    BASE_DIR / "videos-preparados.snappy (1).parquet",
]

VIDEO_PATH = next((p for p in possiveis_videos if p.exists()), None)

if VIDEO_PATH is None:
    raise FileNotFoundError(
        "Não encontrei o arquivo videos-preparados.snappy.parquet na pasta do notebook."
    )

# Leitura do Parquet no DataFrame solicitado.
df_video = spark.read.parquet(str(VIDEO_PATH))

print("Arquivo de vídeos:", VIDEO_PATH)
print("Quantidade de registros:", df_video.count())
df_video.printSchema()


## 2. Leitura do `video-comments-tratados.snappy.parquet`

Agora vamos carregar a base de vídeos e comentários no DataFrame `df_comments`.


In [ ]:
possiveis_comments = [
    BASE_DIR / "video-comments-tratados.snappy.parquet",
    BASE_DIR / "video-comments-tratados.snappy (1).parquet",
    BASE_DIR / "videos-comments-tratados.snappy.parquet",
    BASE_DIR / "videos-comments-tratados.snappy (1).parquet",
]

COMMENTS_PATH = next((p for p in possiveis_comments if p.exists()), None)

if COMMENTS_PATH is None:
    raise FileNotFoundError(
        "Não encontrei o arquivo video-comments-tratados.snappy.parquet na pasta do notebook."
    )

# Leitura da base de comentários.
df_comments = spark.read.parquet(str(COMMENTS_PATH))

print("Arquivo de comentários:", COMMENTS_PATH)
print("Quantidade de registros:", df_comments.count())
df_comments.printSchema()


## 3. Criação das tabelas temporárias

Para usar o Spark SQL, primeiro registramos os DataFrames como **views temporárias**.


In [ ]:
# Registramos a base de vídeos para poder consultá-la com Spark SQL.
df_video.createOrReplaceTempView("tb_video_temp")

# Registramos a base de comentários para poder consultá-la com Spark SQL.
df_comments.createOrReplaceTempView("tb_comments_temp")

# SHOW VIEWS confirma que as views temporárias foram criadas.
spark.sql("SHOW VIEWS").show(truncate=False)


## 4. Join simples usando Spark SQL

Primeiro fazemos o join da forma mais direta possível, usando `Video ID` como chave.

Selecionamos todas as colunas de vídeos e as colunas de comentários, exceto o segundo `Video ID`, para evitar duas colunas com o mesmo nome.


In [ ]:
# Montamos dinamicamente as colunas da base de comentários,
# retirando Video ID porque a chave já estará presente em v.*.
comment_columns_for_sql = [
    f"c.`{coluna}`"
    for coluna in df_comments.columns
    if coluna != "Video ID"
]

# O SELECT final mantém todas as colunas da base de vídeos
# e acrescenta as informações disponíveis da base de comentários.
select_columns = ["v.*"] + comment_columns_for_sql

# Montamos a consulta SQL do join.
sql_join = f"""
SELECT {", ".join(select_columns)}
FROM tb_video_temp v
INNER JOIN tb_comments_temp c
    ON v.`Video ID` = c.`Video ID`
"""

# Executamos a consulta e armazenamos o resultado no DataFrame solicitado.
join_video_comments = spark.sql(sql_join)

print("Resultado do join simples:")
print("Quantidade de registros:", join_video_comments.count())
join_video_comments.show(5, truncate=False)


## 5. Comparação usando `repartition` e `coalesce`

Aqui começamos a olhar para a forma como os dados são distribuídos.

**`repartition`** redistribui os dados e pode executar shuffle. Como o join usa `Video ID`, vamos particionar as duas bases pela mesma chave.

**`coalesce`** reduz a quantidade de partições sem fazer um shuffle completo. Por isso, ele é mais interessante na redução de partições antes da escrita do resultado.


In [ ]:
# Definimos um número pequeno de partições para facilitar os testes em um computador local.
NUM_PARTITIONS = 8

# Reparticionamos os vídeos usando a chave do join.
df_video_repartitioned = df_video.repartition(NUM_PARTITIONS, "Video ID")

# Reparticionamos os comentários usando a mesma chave.
df_comments_repartitioned = df_comments.repartition(NUM_PARTITIONS, "Video ID")

# Registramos as versões reparticionadas como novas views temporárias.
df_video_repartitioned.createOrReplaceTempView("tb_video_repartitioned")
df_comments_repartitioned.createOrReplaceTempView("tb_comments_repartitioned")

# Fazemos novamente o join em SQL usando as bases reparticionadas.
sql_join_repartitioned = f"""
SELECT {", ".join(select_columns)}
FROM tb_video_repartitioned v
INNER JOIN tb_comments_repartitioned c
    ON v.`Video ID` = c.`Video ID`
"""

join_video_comments_repartitioned = spark.sql(sql_join_repartitioned)

print("=== Resultado após repartition ===")
print("Partições dos vídeos:", df_video_repartitioned.rdd.getNumPartitions())
print("Partições dos comentários:", df_comments_repartitioned.rdd.getNumPartitions())
print("Registros:", join_video_comments_repartitioned.count())

# Aplicamos coalesce apenas ao resultado.
# O objetivo é reduzir o número de partições antes da gravação, evitando muitos arquivos pequenos.
join_video_comments_coalesced = join_video_comments_repartitioned.coalesce(4)

print("Partições depois do coalesce:", join_video_comments_coalesced.rdd.getNumPartitions())


# 6. Join otimizado

Agora vamos aplicar as boas práticas estudadas no módulo.

### O que será feito

1. Remover `Video ID` nulo antes do join.
2. Selecionar apenas colunas relevantes.
3. Reparticionar as duas bases pela chave.
4. Criar views temporárias.
5. Identificar a menor base e usar `broadcast`.
6. Consultar o plano com `explain`.

Os comentários foram colocados diretamente no código para explicar cada ação, como solicitado na atividade.


In [ ]:
# ============================================================
# 6.1 FILTRO ANTECIPADO
# ============================================================

# Removemos registros sem Video ID da base de vídeos antes do join.
# Isso reduz dados que nunca poderiam encontrar uma correspondência.
df_video_opt = df_video.filter(F.col("Video ID").isNotNull())

# Fazemos a mesma limpeza na base de comentários.
df_comments_opt = df_comments.filter(F.col("Video ID").isNotNull())

# ============================================================
# 6.2 SELEÇÃO DE COLUNAS
# ============================================================

# Definimos as colunas mais úteis da base de vídeos para o resultado.
video_candidates = [
    "Video ID",
    "Title",
    "Keyword",
    "Published At",
    "Year",
    "Month",
    "Likes",
    "Views",
    "Comments",
    "Interaction",
]

# Mantemos apenas as colunas que realmente existem.
video_cols = [c for c in video_candidates if c in df_video_opt.columns]

# Definimos colunas relevantes da base de comentários.
comment_candidates = [
    "Video ID",
    "Likes Comment",
    "Sentiment",
    "Comment",
    "Comment ID",
]

# Mantemos apenas as colunas disponíveis na base recebida.
comment_cols = [c for c in comment_candidates if c in df_comments_opt.columns]

# A chave precisa estar presente nas duas bases.
if "Video ID" not in video_cols or "Video ID" not in comment_cols:
    raise KeyError(
        "A coluna 'Video ID' precisa existir nas duas bases para realizar o join."
    )

# Selecionamos somente os campos definidos acima.
df_video_opt = df_video_opt.select(*video_cols)
df_comments_opt = df_comments_opt.select(*comment_cols)

print("Colunas de vídeo utilizadas:", video_cols)
print("Colunas de comentários utilizadas:", comment_cols)

# ============================================================
# 6.3 REPARTITION PELA CHAVE
# ============================================================

# Distribuímos os dados das duas bases usando a chave do join.
df_video_opt = df_video_opt.repartition(NUM_PARTITIONS, "Video ID")
df_comments_opt = df_comments_opt.repartition(NUM_PARTITIONS, "Video ID")

# ============================================================
# 6.4 TABELAS TEMPORÁRIAS
# ============================================================

# Criamos views das bases já filtradas, reduzidas e particionadas.
df_video_opt.createOrReplaceTempView("tb_video_opt")
df_comments_opt.createOrReplaceTempView("tb_comments_opt")

# ============================================================
# 6.5 BROADCAST
# ============================================================

# Contamos os registros para identificar qual das duas bases é menor.
# A menor base é uma candidata natural a broadcast.
qtd_video_opt = df_video_opt.count()
qtd_comments_opt = df_comments_opt.count()

print("Vídeos após filtro:", qtd_video_opt)
print("Comentários após filtro:", qtd_comments_opt)

# Se a base de vídeos for menor, ela receberá o hint BROADCAST.
# Caso contrário, o broadcast será aplicado à base de comentários.
broadcast_alias = "v" if qtd_video_opt <= qtd_comments_opt else "c"

# Retiramos Video ID das colunas de comentários para não duplicar a chave no resultado.
comment_columns_opt = [
    f"c.`{coluna}`"
    for coluna in df_comments_opt.columns
    if coluna != "Video ID"
]

# Montamos o conjunto final de colunas.
select_columns_opt = ["v.*"] + comment_columns_opt

# ============================================================
# 6.6 JOIN OTIMIZADO
# ============================================================

# O hint BROADCAST informa ao Spark qual tabela pequena pode ser replicada
# nos nós do cluster para reduzir a movimentação de dados no join.
sql_join_otimizado = f"""
SELECT /*+ BROADCAST({broadcast_alias}) */
       {", ".join(select_columns_opt)}
FROM tb_video_opt v
INNER JOIN tb_comments_opt c
    ON v.`Video ID` = c.`Video ID`
"""

# Executamos o join otimizado usando Spark SQL.
join_otimizado = spark.sql(sql_join_otimizado)

# Mostramos uma amostra para validar o resultado.
print("=== Join otimizado ===")
join_otimizado.show(5, truncate=False)

# Conferimos a quantidade de registros produzidos pelo join.
print("Registros no join otimizado:", join_otimizado.count())


## 6.7 `Explain` — observando os planos de execução

O `explain("formatted")` mostra como o Spark pretende executar cada consulta.

A comparação ajuda a enxergar como mudanças na preparação dos dados podem alterar o plano de execução.


In [ ]:
# Mostra o plano de execução do join original.
print("===== PLANO DO JOIN ORIGINAL =====")
join_video_comments.explain("formatted")

# Mostra o plano do join após repartition.
print("\n===== PLANO DO JOIN COM REPARTITION =====")
join_video_comments_repartitioned.explain("formatted")

# Mostra o plano do join com filtro, seleção de colunas e broadcast.
print("\n===== PLANO DO JOIN OTIMIZADO =====")
join_otimizado.explain("formatted")


# 7. Salvando o resultado otimizado

Nesta última etapa vamos:

1. reduzir as partições do resultado com `coalesce`;
2. gravar o DataFrame em formato Parquet;
3. ler novamente o arquivo;
4. validar quantidade de registros, esquema e alguns exemplos.

O uso de `coalesce` aqui é intencional: queremos reduzir a quantidade de arquivos produzidos na escrita sem fazer um novo shuffle completo.

Os comentários no código explicam cada ação realizada, conforme solicitado.


In [ ]:
# ============================================================
# 7.1 CAMINHO DA SAÍDA
# ============================================================

# Definimos o nome da pasta de saída exatamente como solicitado na atividade.
OUTPUT_PATH = BASE_DIR / "join-videos-comments-otimizado"

# ============================================================
# 7.2 COALESCE ANTES DA ESCRITA
# ============================================================

# Reduzimos o número de partições apenas na etapa final.
# Isso tende a reduzir a quantidade de arquivos gerados pelo Parquet.
join_otimizado_para_gravacao = join_otimizado.coalesce(4)

# ============================================================
# 7.3 ESCRITA EM PARQUET
# ============================================================

# Salvamos o resultado otimizado no formato Parquet.
# overwrite permite executar o notebook novamente sem conflito com uma saída anterior.
(
    join_otimizado_para_gravacao.write
    .mode("overwrite")
    .parquet(str(OUTPUT_PATH))
)

print("Resultado salvo em:", OUTPUT_PATH)

# ============================================================
# 7.4 VALIDAÇÃO DO ARQUIVO GERADO
# ============================================================

# Lemos novamente o Parquet que acabou de ser salvo.
df_join_otimizado_validacao = spark.read.parquet(str(OUTPUT_PATH))

# Confirmamos a quantidade de registros gravados.
print("Registros gravados:", df_join_otimizado_validacao.count())

# Conferimos o esquema armazenado pelo Parquet.
df_join_otimizado_validacao.printSchema()

# Mostramos alguns registros para confirmar visualmente o resultado.
df_join_otimizado_validacao.show(5, truncate=False)


# ✅ Conclusão

Nesta etapa, o projeto foi além de um simples `join`: aplicamos técnicas de otimização para tornar o processamento mais consciente.

Foram praticados:

- filtros antes do join;
- seleção de somente as colunas necessárias;
- `repartition` pela chave `Video ID`;
- `broadcast` da menor base;
- `coalesce` antes da escrita;
- análise de planos de execução com `explain`;
- salvamento do resultado otimizado em Parquet.

A ideia principal é entender que, em grandes volumes de dados, a maneira como os dados são distribuídos e quais informações realmente entram no processamento podem influenciar diretamente a eficiência do Spark.
